# OpenMEP — Google Colab Launcher
**Open-Source MEP Engineering Calculation Suite · v0.2.1**

Runs the **complete full stack** in Google Colab: 29 Streamlit pages, 26 calculation modules,
4 regions (GCC/UAE, Europe/UK, India, Australia), 40+ API endpoints, and all 7 Platform Features
(Project Workspace, Version History, Submission Packager, BIM Bridge, Value Engineering, Company Branding).

## How to run
Click **Runtime → Run all** (or press `Ctrl+F9`) and wait ~90 seconds.
A public URL will be printed at the end — click it to open OpenMEP in your browser.

---
GitHub: https://github.com/kakarot-oncloud/openmep-suite  |  Made by Luquman A

In [ ]:
# Step 1: Clone OpenMEP repository
!git clone https://github.com/kakarot-oncloud/openmep-suite.git
%cd openmep
print('✅ Repository cloned — v0.2.1')

In [ ]:
# Step 2: Install Python dependencies
!pip install -q -r requirements.txt
print('✅ Python dependencies installed')

In [ ]:
# Step 3: Install Node.js 20 and Node.js dependencies (required for Platform Features)
import subprocess
print('Installing Node.js 20...')
subprocess.run(
    ['bash', '-c', 'curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs -q'],
    check=True, capture_output=True
)
result = subprocess.run(['node', '--version'], capture_output=True, text=True)
print(f'Node.js {result.stdout.strip()} installed')
print('Installing npm dependencies...')
subprocess.run(['npm', 'install', '--prefix', 'src'], check=True, capture_output=True)
print('✅ Node.js dependencies installed')

In [ ]:
# Step 4: Start FastAPI backend in background
import subprocess, time, requests, os

os.makedirs('data/projects', exist_ok=True)

api_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
for i in range(20):
    time.sleep(2)
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print('✅ FastAPI backend running — 40+ calculation endpoints ready')
            break
    except Exception:
        pass
else:
    print('⚠️  FastAPI taking longer than expected — continuing anyway')

In [ ]:
# Step 5: Start Node.js Project Management API in background
import subprocess, time, requests, os

env = os.environ.copy()
env['PORT'] = '8080'
env['PROJECTS_DIR'] = os.path.abspath('data/projects')
env['LOG_LEVEL'] = 'silent'

node_proc = subprocess.Popen(
    ['node', '--import', 'tsx/esm', 'index.ts'],
    cwd='src', env=env,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
for i in range(15):
    time.sleep(2)
    try:
        r = requests.get('http://localhost:8080/health', timeout=2)
        if r.status_code == 200:
            print('✅ Node.js API running — project workspaces, versioning, branding ready')
            break
    except Exception:
        pass
else:
    print('⚠️  Node.js API taking longer than expected — continuing anyway')

In [ ]:
# Step 6: Verify all 4 regions are working
import requests, json

regions = [
    ('GCC/UAE (BS 7671)', 'gcc', 40),
    ('Europe/UK (BS 7671)', 'europe', 25),
    ('India (IS 3961)', 'india', 40),
    ('Australia (AS/NZS 3008)', 'australia', 40),
]
for name, region, temp in regions:
    try:
        r = requests.post('http://localhost:8000/api/electrical/cable-sizing', json={
            'region': region, 'load_kw': 45, 'power_factor': 0.85, 'phases': 3,
            'cable_type': 'XLPE_CU', 'installation_method': 'C',
            'cable_length_m': 80, 'ambient_temp_c': temp
        }, timeout=10)
        if r.status_code == 200:
            result = r.json()
            print(f'✅ {name}: {result.get("recommended_size_mm2", "N/A")} mm² cable')
        else:
            print(f'⚠️  {name}: HTTP {r.status_code}')
    except Exception as e:
        print(f'❌ {name}: {e}')

In [ ]:
# Step 7: Launch Streamlit with a public URL
import subprocess, time, os

# Install localtunnel for public URL generation
subprocess.run(['npm', 'install', '-g', 'localtunnel'], capture_output=True)

env = os.environ.copy()
env['API_BASE'] = 'http://localhost:8000'
env['NODE_API_BASE'] = 'http://localhost:8080'

streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app/app.py',
     '--server.port', '8501',
     '--server.address', '0.0.0.0',
     '--server.headless', 'true',
     '--browser.gatherUsageStats', 'false'],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(6)

# Start localtunnel to get a public URL
tunnel_proc = subprocess.Popen(['lt', '--port', '8501'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)
url_line = tunnel_proc.stdout.readline().decode().strip()

print('\n' + '=' * 60)
print('🌐 OpenMEP is ready!')
print(f'📊 Streamlit UI:  {url_line or "http://localhost:8501"}')
print('⚡ FastAPI docs: http://localhost:8000/docs')
print('🗂️  Node.js API:  http://localhost:8080')
print('=' * 60)
print('Session runs up to 12 hours (Google free tier). Data is lost when session ends.')
print('To persist projects: mount Google Drive and set PROJECTS_DIR=/content/drive/MyDrive/openmep-projects')